# Ticket 10: backfill 2024-02 and 2024-03

Checking what P02/P03 actually look like before touching the loader -
row counts, whether `map_account.csv` already covers them (it should,
ticket 3's mapping scope was always P01-P03), and whether the known
`local_amount` defect (#13) is a P01-only artifact or something bigger.

In [1]:
import duckdb, csv

con = duckdb.connect("../warehouse.duckdb", read_only=True)

In [2]:
for p in (2, 3):
    scope = f"company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}"
    print(f"P0{p}:", con.execute(f"SELECT COUNT(*), COUNT(DISTINCT document_id) FROM stg_gl WHERE {scope}").fetchall())

P02: [(13723, 3713)]
P03: [(13521, 3708)]


P02: 13,723 rows / 3,713 documents. P03: 13,521 rows / 3,708 documents.
Both a similar size to P01's 13,142 rows.

In [3]:
with open("../map_account.csv", newline="") as f:
    mapped = {int(r["source_account"]) for r in csv.DictReader(f)}
for p in (2, 3):
    scope = f"company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}"
    accts = {r[0] for r in con.execute(f"SELECT DISTINCT gl_account FROM stg_gl WHERE {scope}").fetchall()}
    print(f"P0{p}: {len(accts)} accounts, {len(accts - mapped)} missing from map_account.csv")

P02: 502 accounts, 0 missing from map_account.csv
P03: 503 accounts, 0 missing from map_account.csv


**0 missing for both periods.** `map_account.csv` was built against the
full P01-P03 scope in ticket 3, not just P01 - no new mapping work
needed for the backfill.

In [4]:
for p in (2, 3):
    scope = f"company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}"
    unbalanced = con.execute(f"""
        SELECT COUNT(*) FROM (SELECT document_id FROM stg_gl WHERE {scope} GROUP BY 1 HAVING ROUND(SUM(debit_amount)-SUM(credit_amount),2)!=0)
    """).fetchone()[0]
    post_close = con.execute(f"SELECT SUM(is_post_close::int) FROM stg_gl WHERE {scope}").fetchone()[0]
    boundary_docs = con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {scope} AND document_type IN ('OPENING_BALANCE','CL')").fetchone()[0]
    print(f"P0{p}: unbalanced_document={unbalanced}, is_post_close={post_close}, opening_balance/closing_entry docs={boundary_docs}")

P02: unbalanced_document=0, is_post_close=62, opening_balance/closing_entry docs=0
P03: unbalanced_document=0, is_post_close=101, opening_balance/closing_entry docs=0


**0 unbalanced documents in either period** (P01 had exactly 1). No
`OPENING_BALANCE`/`CL` documents in P02 or P03 either - expected,
brought-forward balances land at the start of a fiscal year, not mid-year.
`is_post_close` still fires (62 in P02, 101 in P03).

In [5]:
# the big question: is the local_amount defect (issue #13) a P01-only artifact?
for p in (1, 2, 3):
    scope = f"company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}"
    r = con.execute(f"""
        WITH doc_net AS (
            SELECT document_id, ROUND(SUM(local_amount),2) net, COUNT(*) n
            FROM stg_gl WHERE {scope} GROUP BY 1
        )
        SELECT COUNT(*), ROUND(SUM(net),2) FROM doc_net WHERE ABS(net) > 0.01
    """).fetchone()
    print(f"P0{p}: {r[0]} affected documents, {r[1]:,.2f} total local_amount impact")

P01: 20 affected documents, 97,144,587.10 total local_amount impact
P02: 27 affected documents, 104,111,412.57 total local_amount impact
P03: 36 affected documents, 340,498,930.13 total local_amount impact


**No.** It's systemic, and it grows: P01 20 docs / 97,144,587.1, P02 27
docs / 104,111,412.57, P03 36 docs / 340,498,930.13. Same signature as
P01's finding (checked the top 3 offenders per period below): unusually
high line counts, a document total broadcast across every debit line.

In [6]:
for p in (2, 3):
    scope = f"company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}"
    r = con.execute(f"""
        WITH doc_net AS (
            SELECT document_id, document_type, ROUND(SUM(local_amount),2) net, COUNT(*) n
            FROM stg_gl WHERE {scope} GROUP BY 1, 2
        )
        SELECT document_id, document_type, n, net FROM doc_net WHERE ABS(net) > 0.01 ORDER BY ABS(net) DESC LIMIT 3
    """).fetchall()
    print(f"P0{p} top offenders (id, type, line_count, net):", r)

P02 top offenders (id, type, line_count, net): [('fed5dfee-1356-8c59-158b-68354df5bb92', 'HR', 41, 63123588.45), ('24984c80-7a65-8385-083d-62a0fc5dc9d1', 'KR', 50, 9591636.0), ('355cfcba-46ed-8553-2e09-a1a4591d5fe1', 'HR', 79, 6620841.15)]
P03 top offenders (id, type, line_count, net): [('ce738e9e-bc01-8a8e-2e7e-c5b3af381500', 'SA', 56, 219719072.88), ('4626830c-92c7-850c-25bc-9dbd3ac580cb', 'DR', 42, 40296979.16), ('62aa5557-f3c5-85cf-1d9c-5eeaf7a8e078', 'KR', 45, 37795774.17)]


## What I've got

Nothing blocks the backfill mechanically: no new mapping gaps, no
unbalanced documents to design around, no new document-boundary types.
The one thing that changes the picture: the `local_amount` defect is
worse the further into the year the data goes
(97M &rarr; 104M &rarr; 340M), which matters for issue #13's scope (not
P01-specific) and for how P02/P03's period reports word their own
known-issue disclosure (mission 09's pattern, different numbers).